# 🧬 Generative AI for Medical Imaging & Drug Discovery
### By Vivek Nagappa | AI/ML Engineer | Bengaluru, India

---

**Pipeline:**
1. 📦 Setup & Data Loading (MedMNIST PathMNIST)
2. 🎨 Conditional GAN Training (synthetic image generation)
3. 📊 GAN Evaluation (FID score + visual inspection)
4. 🔬 CNN Feature Extractor Training (ResNet-18)
5. 🧬 Biomarker Extraction + t-SNE Visualisation
6. 💊 Drug–Biomarker Correlation Analysis (RDKit)
7. 📈 Results & Drug Candidate Ranking

> **Runtime:** ~45–60 min on Colab T4 GPU  
> **GPU:** Go to Runtime → Change runtime type → T4 GPU

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Install dependencies
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q medmnist rdkit-pypi tqdm scikit-learn torchvision
print('✓ Dependencies installed')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Clone repo (or mount Drive)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

# Option A: Clone from GitHub (replace with your repo URL after pushing)
# !git clone https://github.com/YOUR_USERNAME/genai-medical-imaging.git
# %cd genai-medical-imaging

# Option B: Upload the project folder to Google Drive and mount
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/genai_medical_project

# Option C: Run all code inline in this notebook (default — no repo needed)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/synthetic', exist_ok=True)
os.makedirs('models/gan', exist_ok=True)
os.makedirs('models/cnn', exist_ok=True)
print('✓ Directories created')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Imports & device setup
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torchvision import transforms, models as tv_models
import medmnist
from medmnist import PathMNIST
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
from scipy import stats
from rdkit import Chem
from rdkit.Chem import AllChem
import warnings; warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(42)
np.random.seed(42)

# ── Plot style ──
plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#111820',
                     'axes.edgecolor': '#2d3748', 'grid.color': '#1f2937',
                     'font.family': 'monospace'})
PALETTE = ['#00e5a0','#0099ff','#ff6b35','#a855f7','#f59e0b','#ec4899','#14b8a6','#6366f1','#84cc16']
CLASSES = ['Adipose','Background','Debris','Lymphocytes','Mucus',
           'Smooth Muscle','Normal Colon','Cancer Stroma','Colorectal Adenocarcinoma']
print('✓ Setup complete')

## 📦 Step 1 — Data Loading

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Load MedMNIST PathMNIST dataset
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
IMG_SIZE   = 64
BATCH_SIZE = 64
NUM_CLASSES = 9

def get_transform(augment=False):
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    if augment:
        ops += [transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2)]
    ops += [transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)]
    return transforms.Compose(ops)

class MedDataset(torch.utils.data.Dataset):
    def __init__(self, split, augment=False):
        self.ds = PathMNIST(split=split, transform=get_transform(augment),
                            download=True, root='data/raw', size=IMG_SIZE)
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        img, lbl = self.ds[i]
        return img, lbl.squeeze().long()

train_ds = MedDataset('train', augment=True)
val_ds   = MedDataset('val')
test_ds  = MedDataset('test')

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

# Visualise sample images
imgs, lbls = next(iter(train_loader))
fig, axes = plt.subplots(3, 9, figsize=(18, 6))
for cls in range(9):
    cls_idxs = (lbls == cls).nonzero(as_tuple=True)[0]
    for row in range(min(3, len(cls_idxs))):
        img = imgs[cls_idxs[row]]
        img_disp = np.clip((img.permute(1,2,0).numpy() + 1) / 2, 0, 1)
        axes[row, cls].imshow(img_disp)
        axes[row, cls].axis('off')
        if row == 0:
            axes[row, cls].set_title(CLASSES[cls][:10], fontsize=7, color='#e8edf2')
plt.suptitle('PathMNIST Sample Images (9 classes × 3 samples)', color='#00e5a0', fontsize=12)
plt.tight_layout()
plt.show()

## 🎨 Step 2 — Conditional GAN Training

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Define Generator & Discriminator
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LATENT_DIM = 128

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(NUM_CLASSES, LATENT_DIM)
        self.init_size = IMG_SIZE // 16  # 4
        self.proj = nn.Sequential(
            nn.Linear(LATENT_DIM * 2, 512 * self.init_size * self.init_size),
            nn.LeakyReLU(0.2, True)
        )
        self.conv = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128,  64, 4, 2, 1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.ConvTranspose2d( 64,   3, 4, 2, 1, bias=False), nn.Tanh()
        )
    def forward(self, z, labels):
        x = torch.cat([z, self.label_emb(labels)], dim=1)
        x = self.proj(x).view(-1, 512, self.init_size, self.init_size)
        return self.conv(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(NUM_CLASSES, IMG_SIZE * IMG_SIZE)
        self.model = nn.Sequential(
            nn.Conv2d(4,   64,  4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64,  128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.Conv2d(512, 1,   4, 1, 0, bias=False), nn.Sigmoid()
        )
    def forward(self, img, labels):
        lmap = self.label_emb(labels).view(-1, 1, IMG_SIZE, IMG_SIZE)
        return self.model(torch.cat([img, lmap], dim=1)).view(-1)

def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.constant_(m.bias, 0)

G = Generator().to(DEVICE); G.apply(weights_init)
D = Discriminator().to(DEVICE); D.apply(weights_init)

opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce   = nn.BCELoss()

total_G = sum(p.numel() for p in G.parameters())
total_D = sum(p.numel() for p in D.parameters())
print(f'Generator params:     {total_G:,}')
print(f'Discriminator params: {total_D:,}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — Train cGAN
# Tip: start with 30 epochs for quick results, go to 100 for quality
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GAN_EPOCHS = 50   # Change to 100 for better quality
g_losses, d_losses = [], []

for epoch in range(1, GAN_EPOCHS + 1):
    epoch_g, epoch_d = [], []
    for real_imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{GAN_EPOCHS}', leave=False):
        real_imgs, labels = real_imgs.to(DEVICE), labels.to(DEVICE)
        B = real_imgs.size(0)

        # ── Discriminator ──
        opt_D.zero_grad()
        real_lbl = torch.ones(B, device=DEVICE)  * 0.9   # label smoothing
        fake_lbl = torch.zeros(B, device=DEVICE) + 0.1
        z    = torch.randn(B, LATENT_DIM, device=DEVICE)
        rl   = torch.randint(0, NUM_CLASSES, (B,), device=DEVICE)
        fake = G(z, rl).detach()
        loss_D = (bce(D(real_imgs, labels), real_lbl) + bce(D(fake, rl), fake_lbl)) / 2
        loss_D.backward(); opt_D.step()

        # ── Generator ──
        opt_G.zero_grad()
        z    = torch.randn(B, LATENT_DIM, device=DEVICE)
        rl   = torch.randint(0, NUM_CLASSES, (B,), device=DEVICE)
        fake = G(z, rl)
        loss_G = bce(D(fake, rl), torch.ones(B, device=DEVICE))
        loss_G.backward(); opt_G.step()

        epoch_g.append(loss_G.item()); epoch_d.append(loss_D.item())

    g_losses.append(np.mean(epoch_g))
    d_losses.append(np.mean(epoch_d))
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | G: {g_losses[-1]:.4f} | D: {d_losses[-1]:.4f}')
        torch.save({'epoch': epoch, 'G_state': G.state_dict(), 'D_state': D.state_dict(),
                    'history': {'g_loss': g_losses, 'd_loss': d_losses}},
                   f'models/gan/checkpoint_epoch{epoch}.pt')

print('\n✓ GAN Training complete!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — Visualise GAN results
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Loss curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(g_losses, color='#00e5a0', lw=2, label='Generator')
ax.plot(d_losses, color='#0099ff', lw=2, label='Discriminator')
ax.fill_between(range(len(g_losses)), g_losses, alpha=0.1, color='#00e5a0')
ax.fill_between(range(len(d_losses)), d_losses, alpha=0.1, color='#0099ff')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('cGAN Training Loss', color='#e8edf2', fontsize=13)
ax.legend(); ax.grid(True, ls='--', alpha=0.4)
plt.tight_layout(); plt.show()

# Synthetic image grid
G.eval()
fig, axes = plt.subplots(9, 5, figsize=(12, 20))
with torch.no_grad():
    for cls in range(9):
        z = torch.randn(5, LATENT_DIM, device=DEVICE)
        l = torch.full((5,), cls, dtype=torch.long, device=DEVICE)
        imgs_gen = G(z, l).cpu().numpy()
        for j, img in enumerate(imgs_gen):
            img_d = np.clip((img.transpose(1,2,0) + 1) / 2, 0, 1)
            axes[cls, j].imshow(img_d); axes[cls, j].axis('off')
            if j == 0:
                axes[cls, j].set_ylabel(CLASSES[cls][:12], rotation=0,
                                        labelpad=90, va='center',
                                        fontsize=8, color='#e8edf2')
plt.suptitle('Synthetic Images per Disease Class (5 samples each)',
             color='#00e5a0', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()
G.train()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — Generate & save synthetic dataset
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NUM_SYNTHETIC = 5000   # 500 per class × 9 classes
per_class     = NUM_SYNTHETIC // NUM_CLASSES
syn_imgs, syn_lbls = [], []

G.eval()
with torch.no_grad():
    for cls in range(NUM_CLASSES):
        z = torch.randn(per_class, LATENT_DIM, device=DEVICE)
        l = torch.full((per_class,), cls, dtype=torch.long, device=DEVICE)
        imgs = G(z, l).cpu().numpy()
        syn_imgs.append(imgs)
        syn_lbls.extend([cls] * per_class)

syn_imgs = np.concatenate(syn_imgs, axis=0).astype(np.float32)
syn_lbls = np.array(syn_lbls, dtype=np.int64)
np.save('data/synthetic/synthetic_images.npy', syn_imgs)
np.save('data/synthetic/synthetic_labels.npy', syn_lbls)
print(f'✓ Saved {len(syn_imgs):,} synthetic images  {syn_imgs.shape}')
G.train()

## 🔬 Step 3 — CNN Feature Extractor & Classifier

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 9 — Define CNN Encoder (ResNet-18 backbone)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FEATURE_DIM = 256

class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone     = nn.Sequential(*list(backbone.children())[:-1])  # → (B, 512, 1, 1)
        self.feature_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, FEATURE_DIM), nn.BatchNorm1d(FEATURE_DIM),
            nn.ReLU(True), nn.Dropout(0.4)
        )
        self.classifier   = nn.Sequential(
            nn.Linear(FEATURE_DIM, 128), nn.ReLU(True),
            nn.Dropout(0.2), nn.Linear(128, NUM_CLASSES)
        )
    def forward(self, x, return_features=False):
        feats  = self.feature_head(self.backbone(x))
        logits = self.classifier(feats)
        return (logits, feats) if return_features else logits
    def extract_features(self, x):
        with torch.no_grad():
            return self.feature_head(self.backbone(x))

cnn      = CNNEncoder().to(DEVICE)
crit     = nn.CrossEntropyLoss(label_smoothing=0.1)
opt_cnn  = optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
sched    = optim.lr_scheduler.CosineAnnealingLR(opt_cnn, T_max=30, eta_min=1e-6)
params   = sum(p.numel() for p in cnn.parameters())
print(f'CNN parameters: {params:,}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 10 — Train CNN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CNN_EPOCHS = 20   # 30 for better accuracy
cnn_history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_acc = 0.0

def run_epoch(loader, training=True):
    cnn.train() if training else cnn.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            if training: opt_cnn.zero_grad()
            logits = cnn(imgs)
            loss   = crit(logits, lbls)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
                opt_cnn.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == lbls).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total

for epoch in range(1, CNN_EPOCHS + 1):
    tr_l, tr_a = run_epoch(train_loader, True)
    va_l, va_a = run_epoch(val_loader, False)
    sched.step()
    cnn_history['train_loss'].append(tr_l); cnn_history['val_loss'].append(va_l)
    cnn_history['train_acc'].append(tr_a);  cnn_history['val_acc'].append(va_a)
    flag = ''
    if va_a > best_acc:
        best_acc = va_a
        torch.save({'model_state': cnn.state_dict(), 'best_acc': best_acc,
                    'history': cnn_history}, 'models/cnn/best_cnn.pt')
        flag = '  ← best'
    print(f'Epoch {epoch:3d} | Train Acc: {tr_a:.3f} | Val Acc: {va_a:.3f}{flag}')

print(f'\n✓ CNN Training complete. Best Val Acc: {best_acc:.4f} ({best_acc*100:.2f}%)')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 11 — Evaluate on test set + confusion matrix
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ckpt = torch.load('models/cnn/best_cnn.pt', map_location=DEVICE)
cnn.load_state_dict(ckpt['model_state'])
cnn.eval()

all_preds, all_lbls = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='Evaluating'):
        preds = cnn(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds); all_lbls.extend(lbls.numpy())

all_preds = np.array(all_preds); all_lbls = np.array(all_lbls)
test_acc  = (all_preds == all_lbls).mean()
print(f'\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)\n')
print(classification_report(all_lbls, all_preds, target_names=CLASSES))

# Confusion matrix
cm      = confusion_matrix(all_lbls, all_preds)
cm_norm = cm.astype(float) / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
            linewidths=0.3, linecolor='#1f2937')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'CNN Confusion Matrix  (Test Acc: {test_acc*100:.2f}%)',
             color='#e8edf2', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

## 🧬 Step 4 — Biomarker Extraction & t-SNE

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 12 — Extract 256-dim biomarker features
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
cnn.eval()
all_features, all_labels = [], []

with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='Extracting biomarkers'):
        _, feats = cnn(imgs.to(DEVICE), return_features=True)
        all_features.append(feats.cpu().numpy())
        all_labels.extend(lbls.numpy())

features = np.concatenate(all_features, axis=0)
labels   = np.array(all_labels)

import os; os.makedirs('data/processed', exist_ok=True)
np.save('data/processed/biomarker_features.npy', features)
np.save('data/processed/biomarker_labels.npy',   labels)
print(f'✓ Saved biomarker features: {features.shape}   labels: {labels.shape}')

# t-SNE visualisation
print('Running t-SNE ...')
n_viz   = min(2000, len(features))
idx     = np.random.choice(len(features), n_viz, replace=False)
pca_f   = PCA(n_components=50, random_state=42).fit_transform(features[idx])
emb     = TSNE(n_components=2, perplexity=40, random_state=42, verbose=1).fit_transform(pca_f)

fig, ax = plt.subplots(figsize=(12, 9))
for cls in range(NUM_CLASSES):
    mask = labels[idx] == cls
    ax.scatter(emb[mask,0], emb[mask,1], c=PALETTE[cls],
               label=CLASSES[cls], alpha=0.6, s=14, edgecolors='none')
ax.set_title('Biomarker Feature Space — t-SNE', color='#e8edf2', fontsize=13)
ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8,
          framealpha=0.2, markerscale=2)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.grid(True, ls='--', alpha=0.3)
plt.tight_layout(); plt.show()

## 💊 Step 5 — Drug–Biomarker Correlation Analysis

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 13 — Build molecular dataset with RDKit
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MOLECULES = [
    ('Ibuprofen',    'CC(C)Cc1ccc(cc1)C(C)C(=O)O',                         'COX-2',    13000),
    ('Aspirin',      'CC(=O)Oc1ccccc1C(=O)O',                               'COX-1',    50000),
    ('Metformin',    'CN(C)C(=N)NC(=N)N',                                   'AMPK',      1200),
    ('Erlotinib',    'C#Cc1cccc(Nc2ncnc3cc(OCC)c(OCC)cc23)c1',              'EGFR',         2),
    ('Imatinib',     'Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1','BCR-ABL',100),
    ('Gefitinib',    'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',     'EGFR',        33),
    ('Sorafenib',    'CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1','RAF', 10),
    ('Tamoxifen',    'CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1',        'ESR1',      3400),
    ('Doxorubicin',  'COc1cccc2C(=O)c3c(O)c4C(O)(CC(O)(CC(=O)CO)c4c(O)c3=O)c21','TOP2A',200),
    ('Methotrexate', 'CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(cc1)C(=O)NC(CCC(=O)O)C(=O)O','DHFR',5),
    ('Cisplatin',    '[NH3][Pt]([NH3])(Cl)Cl',                              'DNA',       3000),
    ('Paclitaxel',   'O=C(c1ccccc1)N[C@@H](C(=O)O[C@H]2C[C@]3(O)C(=O)[C@H](OC(C)=O)[C@@H]3O2)c1ccccc1','Tubulin',3),
    ('Bortezomib',   'CC(C)C[C@@H](NC(=O)[C@@H](Cc1cccnc1)NC(=O)c1cnccn1)B(O)O','26S-Prot',0.6),
    ('Ibrutinib',    'O=C(/C=C/c1ccccc1)N1CCC[C@@H]1c1ncnc2[nH]ccc12',    'BTK',         0.5),
    ('Ruxolitinib',  'C[C@@H](Cc1cncn1C)Nc1ncnc2[nH]cc(-c3ccncc3)c12',    'JAK1/2',      3.3),
    ('Lenalidomide', 'O=C1CCC(N2C(=O)c3cccc(N)c3C2=O)C(=O)N1',            'CRBN',      1000),
    ('Vemurafenib',  'CCCS(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3cc(ccc23)-c2ccc(Cl)cc2)c1','BRAF',31),
    ('Carboplatin',  'O=C1OC(=O)[C@H]2CCCCC[Pt]12',                       'DNA',       1000),
    ('Thalidomide',  'O=C1CCC(N2C(=O)c3ccccc3C2=O)C(=O)N1',               'CRBN',       100),
    ('Vincristine',  'CCC1(CC)C=C2CN3CCc4cc5c(cc4[C@@H]3C[C@H]2C1)OC(=O)c1[nH]cc(CC)c1CC','Tubulin',50),
]

def morgan_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nbits)
    return np.array(fp, dtype=np.float32)

records = []
for name, smi, target, ic50 in MOLECULES:
    fp = morgan_fp(smi)
    if fp is not None:
        records.append({'name': name, 'smiles': smi, 'target': target,
                        'ic50_nM': ic50, 'pIC50': -np.log10(ic50 * 1e-9), 'fp': fp})
mol_df = pd.DataFrame(records)
print(f'✓ Built molecular dataset: {len(mol_df)} molecules  ×  {mol_df.iloc[0]["fp"].shape[0]}-dim Morgan fingerprint')
print(mol_df[['name','target','pIC50']].to_string(index=False))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 14 — Correlation analysis
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from sklearn.preprocessing import StandardScaler

# Reduce molecular fingerprints via PCA
fp_matrix = np.vstack(mol_df['fp'].values)
fp_scaled = StandardScaler().fit_transform(fp_matrix)
fp_pca    = PCA(n_components=min(19, len(mol_df)-1), random_state=42).fit_transform(fp_scaled)
print(f'Molecular fingerprint PCA: {fp_matrix.shape} → {fp_pca.shape}')

# Per-class mean biomarker profile
class_profiles = {}
for cls in range(NUM_CLASSES):
    mask = labels == cls
    if mask.sum() > 0:
        class_profiles[cls] = features[mask].mean(0)  # (256,)

# Correlate each class profile with each molecule's PCA vector
n_mol_dims = fp_pca.shape[1]
results = {}

for cls_idx, bm_vec in class_profiles.items():
    bm_proj  = bm_vec[:n_mol_dims]   # first n_mol_dims dims of biomarker
    corr_row = []
    for mol_vec in fp_pca:
        r, p = stats.pearsonr(bm_proj, mol_vec)
        corr_row.append((r if np.isfinite(r) else 0.0, p))
    df_cls = mol_df.copy()
    df_cls['correlation'] = [c[0] for c in corr_row]
    df_cls['p_value']     = [c[1] for c in corr_row]
    df_cls['abs_corr']    = df_cls['correlation'].abs()
    df_cls['significant'] = df_cls['p_value'] < 0.05
    df_cls = df_cls.sort_values('abs_corr', ascending=False).reset_index(drop=True)
    results[CLASSES[cls_idx]] = df_cls.drop(columns=['fp'])
    print(f'  {CLASSES[cls_idx]:32s} → Top drug: {df_cls.iloc[0]["name"]:15s}  r={df_cls.iloc[0]["correlation"]:+.3f}')

print('\n✓ Correlation analysis complete!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 15 — Visualise drug candidates
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TARGET_CLASS = 'Colorectal Adenocarcinoma'   # Change to any class
TOP_K        = 10

df_show = results[TARGET_CLASS].head(TOP_K)

fig, ax = plt.subplots(figsize=(11, 5))
colors  = [PALETTE[0] if r > 0 else PALETTE[2] for r in df_show['correlation']]
bars    = ax.barh(df_show['name'], df_show['abs_corr'], color=colors, edgecolor='none', height=0.6)

for bar, (_, row) in zip(bars, df_show.iterrows()):
    ax.text(bar.get_width() + 0.003,
            bar.get_y() + bar.get_height()/2,
            f"r={row['correlation']:+.3f}  [{row['target']}]  pIC50={row['pIC50']:.2f}",
            va='center', fontsize=8.5, color='#e8edf2')

ax.set_xlim(0, df_show['abs_corr'].max() + 0.15)
ax.set_title(f'Top {TOP_K} Drug Candidates — {TARGET_CLASS}',
             color='#e8edf2', fontsize=13)
ax.set_xlabel('|Pearson Correlation|')
ax.invert_yaxis()
ax.grid(True, axis='x', ls='--', alpha=0.4)
plt.tight_layout(); plt.show()

# Full results table
print(f'\nFull results for: {TARGET_CLASS}')
print(df_show[['name','target','correlation','abs_corr','pIC50','ic50_nM','significant']]
      .rename(columns={'abs_corr':'|r|','ic50_nM':'IC50(nM)'})
      .to_string(index=False))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 16 — Drug-Disease Correlation Heatmap
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Build pivot: drugs × disease classes
top5_per_class = []
for cls_name, df_cls in results.items():
    for _, row in df_cls.head(5).iterrows():
        top5_per_class.append({'drug': row['name'], 'class': cls_name,
                               'abs_corr': row['abs_corr']})

summary_df = pd.DataFrame(top5_per_class)
pivot = summary_df.pivot_table(index='drug', columns='class',
                               values='abs_corr', aggfunc='mean').fillna(0)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, cmap='YlGnBu', ax=ax, linewidths=0.3,
            linecolor='#1f2937', fmt='.2f', annot=True, annot_kws={'size': 7})
ax.set_title('Drug–Disease Biomarker Correlation Heatmap',
             color='#e8edf2', fontsize=13, pad=12)
ax.set_xlabel('Disease Class', fontsize=10)
ax.set_ylabel('Drug Compound',  fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 17 — Final summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('=' * 60)
print('   PIPELINE SUMMARY')
print('=' * 60)
print(f'  cGAN epochs trained:      {GAN_EPOCHS}')
print(f'  Synthetic images generated:{len(syn_imgs):,}')
print(f'  CNN test accuracy:          {test_acc*100:.2f}%')
print(f'  Biomarker feature dim:      {features.shape[1]}')
print(f'  Molecules analysed:         {len(mol_df)}')
print(f'  Disease classes covered:    {NUM_CLASSES}')
print(f'  Top candidates per class:   10')
print('=' * 60)
print(f'\n  Top candidate per disease class:')
for cls_name, df_cls in results.items():
    top = df_cls.iloc[0]
    print(f'  {cls_name:32s} → {top["name"]:15s}  r={top["correlation"]:+.3f}')
print('\n✓ Complete pipeline run successful!')